# 04. Trực quan hóa dữ liệu

Notebook này vẽ các biểu đồ khám phá xu hướng sử dụng AI theo thời gian, vai trò của file trong dự án, tỷ lệ code bị sửa trong 30 ngày đầu và mục đích của lần sửa đầu tiên.

Import thư viện, tải dữ liệu đã xử lý từ `data/processed/commits_features.csv` và tạo hàm lưu hình vào `reports/figures/`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = Path("../data/processed/commits_features.csv")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
df["commit_date"] = pd.to_datetime(df["commit_date"], utc=True, errors="coerce")
df["change_commit_date"] = pd.to_datetime(df["change_commit_date"], utc=True, errors="coerce")
df["year_quarter"] = df["commit_date"].dt.to_period("Q").astype(str)

print(f"Loaded {DATA_PATH}: {len(df):,} rows x {df.shape[1]} columns")

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print(f"Saved {path}")
    plt.show()
    plt.close()

Loaded ../data/processed/commits_features.csv: 21,287 rows x 32 columns


## B1. Xu hướng và bối cảnh sử dụng AI

Xem tốc độ tăng trưởng số lượng khối code AI theo tháng và sự dịch chuyển thị phần giữa các công cụ AI qua từng quý.

In [2]:
monthly = df.groupby("year_month").size().sort_index()
plt.figure(figsize=(13, 4.5))
plt.plot(monthly.index, monthly.values, marker="o", linewidth=1.8, color="#2563eb")
plt.xticks(rotation=60)
plt.title("So khoi code co comment AI theo thang")
plt.xlabel("Thang")
plt.ylabel("So khoi code")
savefig("B1_01_monthly_growth.png")

tool_q = pd.crosstab(df["year_quarter"], df["ai_tool"]).sort_index()
tool_q_share = tool_q.div(tool_q.sum(axis=1), axis=0) * 100
ax = tool_q_share.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="tab10", width=0.82)
ax.set(title="Ty le cac cong cu AI theo quy (%)", xlabel="Quy", ylabel="Ty le (%)")
ax.legend(title="ai_tool", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=45)
savefig("B1_02_tool_share_by_quarter.png")

Saved ../reports/figures/B1_01_monthly_growth.png
Saved ../reports/figures/B1_02_tool_share_by_quarter.png


So sánh tỷ lệ giữa Python và JavaScript, đồng thời xem phân bố các công cụ AI và mức độ hỗ trợ (`written` so với `assisted`).

In [3]:
lang = df["language"].value_counts()
plt.figure(figsize=(5.5, 5.5))
plt.pie(
    lang.values,
    labels=[f"{k}: {v:,}" for k, v in lang.items()],
    autopct="%.1f%%",
    startangle=90,
    colors=["#3776AB", "#F7DF1E"],
)
plt.title("Ty le ngon ngu lap trinh")
savefig("B1_03_language_share.png")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
tool_counts = df["ai_tool"].value_counts()
sns.barplot(x=tool_counts.values, y=tool_counts.index, ax=axes[0], color="#0891b2")
axes[0].set(title="Phan bo cong cu AI", xlabel="So khoi code", ylabel="ai_tool")

verb_counts = df["ai_verb"].value_counts()
sns.barplot(x=verb_counts.values, y=verb_counts.index, ax=axes[1], color="#f97316")
axes[1].set(title="Muc do dong gop (ai_verb)", xlabel="So khoi code", ylabel="ai_verb")
savefig("B1_04_ai_tool_and_verb.png")

Saved ../reports/figures/B1_03_language_share.png
Saved ../reports/figures/B1_04_ai_tool_and_verb.png


Kiểm tra xem AI đang được dùng nhiều ở loại file nào (`file_role`) theo từng ngôn ngữ và liệt kê top 15 repository có nhiều khối code AI nhất để nhận diện các repo chiếm tỷ trọng lớn.

In [4]:
role_lang = pd.crosstab(df["file_role"], df["language"])
role_lang = role_lang.loc[role_lang.sum(axis=1).sort_values(ascending=True).index]
ax = role_lang.plot(kind="barh", stacked=True, figsize=(9, 4.8), color=["#F7DF1E", "#3776AB"])
ax.set(title="Phan bo vai tro file (file_role) theo ngon ngu", xlabel="So khoi code", ylabel="file_role")
savefig("B1_05_file_role_by_language.png")

top_repo = df["repo"].value_counts().head(15).sort_values()
plt.figure(figsize=(9, 5.5))
sns.barplot(x=top_repo.values, y=top_repo.index, color="#7c3aed")
plt.title("Top 15 repository co nhieu khoi code AI nhat")
plt.xlabel("So khoi code")
plt.ylabel("Repository")
savefig("B1_06_top_repositories.png")

Saved ../reports/figures/B1_05_file_role_by_language.png
Saved ../reports/figures/B1_06_top_repositories.png


## B2. Tỷ lệ chỉnh sửa trong 30 ngày và mục đích sửa đổi

So sánh tỷ lệ code bị sửa trong vòng 30 ngày đầu (`change_30d`) theo từng công cụ AI và từng vai trò file. Chỉ tính trên các khối có đủ 30 ngày theo dõi để tránh sai lệch với dữ liệu mới.

In [5]:
df_30d = df[df["change_30d"].notna()].copy()
df_30d["change_30d"] = df_30d["change_30d"].astype(bool)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)
tool_30 = df_30d.groupby("ai_tool")["change_30d"].mean().mul(100).sort_values()
tool_30.plot(kind="barh", ax=axes[0], color="#16a34a")
axes[0].set(title="Ty le sua trong 30 ngay theo cong cu AI", xlabel="% bi sua trong 30 ngay", ylabel="ai_tool")

role_30 = df_30d.groupby("file_role")["change_30d"].mean().mul(100).sort_values()
role_30.plot(kind="barh", ax=axes[1], color="#2563eb")
axes[1].set(title="Ty le sua trong 30 ngay theo vai tro file", xlabel="% bi sua trong 30 ngay", ylabel="file_role")
savefig("B2_01_change_30d_by_tool_and_role.png")

Saved ../reports/figures/B2_01_change_30d_by_tool_and_role.png


Phân tích mục đích của lần chỉnh sửa đầu tiên (`change_intent`) trên các khối có ghi nhận thay đổi, chia theo vai trò file và công cụ AI để biết code AI bị sửa vì lỗi hay vì cập nhật tính năng.

In [6]:
changed = df[df["has_recorded_change"]].copy()
intent_values = set(changed["change_intent"].unique())

if len(intent_values - {"other_change", "no_recorded_change"}) == 0:
    print("change_intent chi co nhan mac dinh. Hay chay lai notebook 01 va 03 de giu cot change_commit_message.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
    ct_role = pd.crosstab(changed["file_role"], changed["change_intent"])
    ct_role_share = ct_role.div(ct_role.sum(axis=1), axis=0) * 100
    ct_role_share.plot(kind="barh", stacked=True, ax=axes[0], colormap="Set2", width=0.8)
    axes[0].set(title="Muc dich lan sua dau tien theo vai tro file (%)", xlabel="%", ylabel="file_role")
    axes[0].legend(title="change_intent", loc="lower right")

    ct_tool = pd.crosstab(changed["ai_tool"], changed["change_intent"])
    ct_tool_share = ct_tool.div(ct_tool.sum(axis=1), axis=0) * 100
    ct_tool_share.plot(kind="barh", stacked=True, ax=axes[1], colormap="Set2", width=0.8)
    axes[1].set(title="Muc dich lan sua dau tien theo cong cu AI (%)", xlabel="%", ylabel="ai_tool")
    axes[1].legend(title="change_intent", bbox_to_anchor=(1.01, 1), loc="upper left")
    savefig("B2_02_change_intent_breakdown.png")

Saved ../reports/figures/B2_02_change_intent_breakdown.png


## B3. Phân phối và tương quan giữa các biến số

Kiểm tra độ lệch của biến `days_to_modification` và vẽ biểu đồ Q-Q trên thang logarit (`log1p`) để kiểm tra tính chuẩn trước khi chạy hồi quy.

In [7]:
days = df["days_to_modification"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.histplot(days, bins=50, kde=True, ax=axes[0], color="#2563eb")
axes[0].set(title="Phan phoi days_to_modification", xlabel="So ngay den lan sua dau tien", ylabel="So khoi code")

stats.probplot(np.log1p(days), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot log1p(days_to_modification)")
savefig("B3_01_days_distribution_and_qq.png")

Saved ../reports/figures/B3_01_days_distribution_and_qq.png


So sánh kích thước đoạn code (trên thang `log1p(code_lines)`) theo ngôn ngữ và theo vai trò file để tránh bị bẹp biểu đồ do các file quá dài.

In [8]:
df["log_code_lines"] = np.log1p(df["code_lines"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.violinplot(data=df, x="language", y="log_code_lines", ax=axes[0], cut=0, palette=["#3776AB", "#F7DF1E"])
axes[0].set(title="Phan phoi log1p(code_lines) theo ngon ngu", xlabel="Ngon ngu", ylabel="log1p(code_lines)")

sns.boxplot(data=df, x="file_role", y="log_code_lines", ax=axes[1], showfliers=False, color="#0891b2")
axes[1].set(title="Phan phoi log1p(code_lines) theo vai tro file", xlabel="file_role", ylabel="log1p(code_lines)")
axes[1].tick_params(axis="x", rotation=20)
savefig("B3_02_code_lines_by_lang_and_role.png")

Saved ../reports/figures/B3_02_code_lines_by_lang_and_role.png


Vẽ ma trận tương quan giữa các biến số chính trong dữ liệu.

In [9]:
corr_cols = [
    "year",
    "month",
    "is_weekend",
    "has_recorded_change",
    "days_to_modification",
    "code_lines",
    "code_chars",
    "comment_chars",
    "repo_ai_commit_count",
]
available_corr = [c for c in corr_cols if c in df.columns]
corr = df[available_corr].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Ma tran tuong quan giua cac bien so")
savefig("B3_03_correlation_matrix.png")

Saved ../reports/figures/B3_03_correlation_matrix.png
